# Stage 5: Deployment Benchmarking (Inference)

Simulates a deployed NLP sentiment service by running a fixed-size inference workload through every model in `NLP_MODELS` and recording per-query and per-token timing. The goal is to characterize **intrinsic per-token inference cost as a function of architecture**, which together with the energy-join gives the energy-per-token figure that anchors the deployment side of the project's tradeoff curve.

## Independence from Stage 3
This stage **deliberately loads raw HuggingFace checkpoints** (e.g., `bert-base-uncased`) rather than the fine-tuned weights produced by Stage 3. Per-token inference cost is a property of the model architecture (parameter count, attention pattern, hidden width), not of what the parameters happen to encode after fine-tuning. Running on raw checkpoints means Stage 5 is reproducible standalone: it can be re-run without depending on a successful Stage 3 sweep.

## What this stage produces
`./results/inference_results.csv`: one row per model with `n_queries`, `total_tokens`, `time_s`, `time_per_query`, `time_per_token`, and ISO 8601 UTC start/end timestamps for the inference window (used by the external energy join).

## What this stage consumes
- `./artifacts/text_df.parquet` and `./artifacts/num_df.parquet` (Stage 1): text is the inference workload; num is only used to compute the shared cutoff.


In [1]:
# Shared config/helpers and model registry.
from common import *

# Transformer classes for batch inference benchmarking.
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2ForSequenceClassification,
    GPT2Tokenizer,
)
# Wall-clock ISO timestamps for external energy join.
from datetime import datetime, timezone

# Load text data and num data (num is only needed for the shared cutoff).
text_df = load_text_df()
num_df = load_num_df()

# Date normalization for consistent split logic.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# Same chronological cutoff date as other stages.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)
_, text_test_df = chronological_split_by_date(
    df=text_df,
    date_col="date",
    cutoff_date=cutoff_date,
)

# Inference benchmark record collector.
results = ResultsCollector()

[common] device=cuda  artifacts=/home/hrilab/energy-analysis-pipeline/artifacts  results=/home/hrilab/energy-analysis-pipeline/results


## 5.1 Benchmark loop

In [2]:
# Number of requests to benchmark and per-batch size.
n_infer = CONFIG["inference"]["n_samples"]
infer_batch = CONFIG["inference"]["batch_size"]

# Use the chronological test tail in time order (not a random sample).
# A deployed inference service serves queries as they arrive in real time,
# not in a shuffled batch: keeping date order makes the benchmark a closer
# analog to production behavior. Using the test tail (post-cutoff) also
# avoids benchmarking on text the model could have seen during a future
# fine-tune of the same architecture.
infer_texts = text_test_df.sort_values("date")["text"].head(n_infer).tolist()
if not infer_texts:
    raise ValueError("Chronological test split is empty; cannot benchmark inference.")

print(f"Inference sample size from chronological test tail: {len(infer_texts)}")

# Benchmark each model on the same inference text set.
for model_name, model_path in NLP_MODELS.items():
    print(f"\nInference benchmark: {model_name}")

    # GPT-2 uses a custom tokenizer/model pair for padding support.
    is_gpt2 = "gpt2" in model_path
    if is_gpt2:
        tokenizer = GPT2Tokenizer.from_pretrained(model_path)
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2ForSequenceClassification.from_pretrained(
            model_path,
            num_labels=CONFIG["nlp"]["num_labels"],
        )
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=CONFIG["nlp"]["num_labels"],
        )

    # Eval mode and selected device.
    model.to(DEVICE)
    # Note: Stage 5 deliberately benchmarks raw checkpoints (not Stage 3
    # fine-tuned weights) because per-token inference cost is a function of
    # architecture, not trained parameters. This makes Stage 5 reproducible
    # without depending on Stage 3 having run successfully.
    model.eval()

    total_tokens = 0
    # Wrap the entire inference loop with wall-clock anchors for energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    # torch.no_grad() disables autograd's gradient tracking for the duration
    # of this block. Gradients are not needed for forward-only inference, and
    # turning them off saves both memory (no activation tape stored) and time
    # (no graph construction). Standard pattern for any benchmark that should
    # reflect production inference cost.
    with torch.no_grad():
        # Fixed-size batches through the model.
        for i in range(0, len(infer_texts), infer_batch):
            batch_texts = infer_texts[i : i + infer_batch]
            enc = tokenizer(
                batch_texts,
                truncation=True,
                padding="max_length",
                max_length=CONFIG["nlp"]["max_length"],
                return_tensors="pt",
            ).to(DEVICE)

            # numel counts every position in input_ids INCLUDING padding-token
            # positions, since enc was built with padding="max_length". The
            # resulting `time_per_token` is therefore per *padded* token, not
            # per real content token. This is intentional: a deployed system
            # pays the FLOPs cost of padded positions too (transformer compute
            # is shape-driven, not content-driven), so per-padded-token time
            # is the more honest deployment metric.
            total_tokens += enc["input_ids"].numel()
            _ = model(**enc)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # Throughput metrics used by Stage 6/7 analysis.
    record = {
        "model": model_name,
        "n_queries": len(infer_texts),
        "total_tokens": total_tokens,
        "time_s": infer_time,
        "time_per_query": infer_time / len(infer_texts),
        "time_per_token": infer_time / total_tokens,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }
    results.add_inference(record)

    print(
        f"  {len(infer_texts)} queries in {infer_time:.1f}s  |  "
        f"time/query={record['time_per_query']:.4f}s"
    )

    # Release model memory before benchmarking the next checkpoint.
    del model
    torch.cuda.empty_cache()

# Preview benchmark results.
results.inference_df().round(8)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inference sample size from chronological test tail: 1000

Inference benchmark: DistilBERT


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  1000 queries in 0.5s  |  time/query=0.0005s

Inference benchmark: BERT-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  1000 queries in 0.7s  |  time/query=0.0007s

Inference benchmark: RoBERTa-base


  1000 queries in 0.6s  |  time/query=0.0006s

Inference benchmark: FinBERT


  1000 queries in 0.7s  |  time/query=0.0007s


,model,n_queries,total_tokens,time_s,time_per_query,time_per_token,wall_infer_start_iso,wall_infer_end_iso
0,DistilBERT,1000,128000,0.452551,0.000453,0.000004,2026-05-07T22:54:16.677343+00:00,2026-05-07T22:54:17.129910+00:00
1,BERT-base,1000,128000,0.665617,0.000666,0.000005,2026-05-07T22:54:17.348414+00:00,2026-05-07T22:54:18.014042+00:00
2,RoBERTa-base,1000,128000,0.637207,0.000637,0.000005,2026-05-07T22:54:18.241535+00:00,2026-05-07T22:54:18.878753+00:00
3,FinBERT,1000,128000,0.663088,0.000663,0.000005,2026-05-07T22:54:19.287313+00:00,2026-05-07T22:54:19.950414+00:00


## 5.2 Persist results

In [3]:
results.save(RESULTS_DIR)
print(f"Total inference benchmarks: {len(results.inference_results)}")

Results saved to /home/hrilab/energy-analysis-pipeline/results/
Total inference benchmarks: 4
